In [ ]:
import os
import gc
import csv
import time
import yaml
import shutil
import random
import kagglehub
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt 
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

# Utils

### DOWNLOAD=True if you want to import into Input in Kaggle Notebook and download in Google Colab

CHANGE `DATASET_PATH` TO APPROPRIATE PATH IF ON GOOGLE COLAB

https://www.kaggle.com/datasets/sshikamaru/fruit-recognition

In [ ]:
DOWNLOAD = False

#### Insert your username below

In [ ]:
username = ""
MODEL_PATH = Path(f"/kaggle/input/models/{username}/densenet/pytorch/default/1")
CONFIG_PATH = Path(f"/kaggle/input/datasets/{username}/densenet/")
DATASET_PATH = "/kaggle/input/datasets/gpiosenka/butterfly-images40-species"

In [ ]:
def load_yaml(path):
    with open(path, "r") as f:
        cfg = yaml.safe_load(f)
    return cfg

def set_seed(seed: int = 42):
    random.seed(seed)                     # Python random
    np.random.seed(seed)                  # NumPy
    torch.manual_seed(seed)               # CPU
    torch.cuda.manual_seed(seed)          # GPU
    torch.cuda.manual_seed_all(seed)      # All GPUs
    torch.backends.cudnn.deterministic = True  # Deterministic convs
    torch.backends.cudnn.benchmark = False     # Disable auto-tuner for reproducibility
    print(f"Random seed set to {seed}")

def save_training_plots(
    model_name,
    loss_history,
    train_acc_history,
    test_acc_history,
    epoch_times,
    output_dir="outputs/plots"
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    epochs = np.arange(1, len(loss_history) + 1)

    # Loss plot
    plt.figure()
    plt.plot(epochs, loss_history, label="Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training Loss - {model_name}")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / f"{model_name}_loss.png")
    plt.close()

    # Accuracy plot
    plt.figure()
    plt.plot(epochs, train_acc_history, label="Train Accuracy")
    plt.plot(epochs, test_acc_history, label="Test Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"Train vs Test Accuracy - {model_name}")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / f"{model_name}_accuracy.png")
    plt.close()

    # Time per epoch plot
    plt.figure()
    plt.plot(epochs, epoch_times, label="Time per Epoch (s)")
    plt.xlabel("Epoch")
    plt.ylabel("Seconds")
    plt.title(f"Epoch Time - {model_name}")
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / f"{model_name}_epoch_time.png")
    plt.close()

    print(f"\nPlots saved to: {output_dir.resolve()}")

def summarize_checkpoint_times(ckpt_path):
    ckpt = torch.load(MODEL_PATH / ckpt_path, map_location="cpu")
    
    # Check if epoch_times exists
    if "epoch_times" not in ckpt:
        print("Checkpoint does not contain 'epoch_times'.")
        return None

    epoch_times = ckpt["epoch_times"]
    total_time = sum(epoch_times)
    avg_time = total_time / len(epoch_times)

    def format_hms(seconds):
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = int(seconds % 60)
        return f"{h}h {m}m {s}s"

    print(f"Average epoch time: {format_hms(avg_time)}")
    print(f"Total training time: {format_hms(total_time)}")
    
    return avg_time, total_time

def print_model_size(model: nn.Module, device, input_size=(3, 224, 224)):
    """
    Prints the number of parameters and approximate memory size of a PyTorch model.

    Args:
        model (nn.Module): The model to inspect.
        input_size (tuple): Input size (C, H, W), default is (3, 224, 224).
    """
    # Total parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Estimate model size (bytes)
    # float32 = 4 bytes
    size_bytes = total_params * 4
    size_mb = size_bytes / (1024 ** 2)

    print(f"Model: {model.__class__.__name__}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Approximate size: {size_mb:.2f} MB")

    # Measure memory usage
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
    
    dummy_input = torch.randn(1, *input_size).to(device)
    model.eval()
    with torch.no_grad():
        _ = model(dummy_input)

    if device.type == 'cuda':
        mem_alloc = torch.cuda.memory_allocated(device) / (1024 ** 2)
        mem_peak = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
        print(f"GPU Memory Allocated: {mem_alloc:.2f} MB")
        print(f"GPU Peak Memory Allocated: {mem_peak:.2f} MB")

In [ ]:
DENSENET_CFG = load_yaml(CONFIG_PATH / "densenet.yaml")
RESNET_CFG = load_yaml(CONFIG_PATH / "resnet.yaml")
DATA_CFG = load_yaml(CONFIG_PATH / "data.yaml")

# Dataset

In [ ]:
def download_data(data_dir):
    data_dir = Path(BASE_PATH) / data_dir
    data_dir.mkdir(parents=True, exist_ok=True)

    download_path = kagglehub.dataset_download("gpiosenka/butterfly-images40-species")

    print("Path to dataset files:", download_path)

    print(f"Moving data into {data_dir} ...")
    for content_path in Path(download_path).iterdir():
        shutil.move(content_path, data_dir)
    print("Moving complete!")
    return data_dir

In [ ]:
if DOWNLOAD:
    download_data(DATA_CFG['root'])

In [ ]:
def build_transforms(image_size=224, train=True):
    if train:
        return T.Compose([
            T.RandomResizedCrop(image_size),
            T.RandomHorizontalFlip(),
            T.RandomRotation(15),
            T.ToTensor(),
            T.Normalize(mean=DATA_CFG["mean"], std=DATA_CFG["std"])
        ])
    else:
        return T.Compose([
            T.Resize(image_size),
            T.ToTensor(),
            T.Normalize(mean=DATA_CFG["mean"], std=DATA_CFG["std"])
        ])     
    
class ButterflyDataset(Dataset):
    def __init__(self, root, split="train", transform=None):
        self.root = Path(root)
        self.data_dir = self.root / split
        self.transform = transform

        self.df = pd.read_csv(self.root / "butterflies and moths.csv")

        data = self.df[self.df["data set"] == split]
        _unique_id_and_classes = data[["class id", "labels"]].drop_duplicates().reset_index(drop=True)
        self.class_to_idx = {cls_name: id for id, cls_name in _unique_id_and_classes.itertuples(index=False)}
        self.idx_to_class = {id: cls_name for cls_name, id in self.class_to_idx.items()}
        
        self.samples = []
        for _, row in data.iterrows():
            filename = row["filepaths"]
            label_name = row["labels"]
            label_idx = self.class_to_idx[label_name]
            filepath = self.root / filename
            self.samples.append((filepath, label_idx))

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, index):
        filepath, label = self.samples[index]
        image = Image.open(filepath).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label

## Get dataset mean and standard deviation

In [ ]:
MEAN_STD = False

In [ ]:
if MEAN_STD:
    transform = T.Compose([
        T.ToTensor()
    ])
    
    train_datasets = ButterflyDataset(
            root=DATASET_PATH, 
            split="train",
            transform=transform)
    
    loader = DataLoader(train_datasets, batch_size=64, shuffle=False)
    
    mean = torch.zeros(3)
    std = torch.zeros(3)
    total_pixels = 0
    
    for images, _ in tqdm(loader):
        b, c, h, w = images.shape
        num_pixels = b * h * w
    
        mean += images.sum(dim=[0, 2, 3])
        std += (images ** 2).sum(dim=[0, 2, 3])
        total_pixels += num_pixels
    
    mean /= total_pixels
    std = torch.sqrt(std / total_pixels - mean ** 2)
    
    print("Mean:", mean)
    print("Std:", std)

# Model

## DenseNet

In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate=32):
        super(DenseLayer, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_channels, out_channels=4*growth_rate, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(4*growth_rate)
        self.conv2 = nn.Conv2d(4*growth_rate, out_channels=growth_rate, kernel_size=3, stride=1, padding=1, bias=False)
    
    def forward(self, x):
        identity = x

        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv1(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv2(x)

        out = torch.cat((identity, x), dim=1)
        return out

class DenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels, growth_rate=32):
        super(DenseBlock, self).__init__()
        self.num_layers = num_layers
        self.in_channels = in_channels
        self.growth_rate = growth_rate

        layers = []
        for _ in range(num_layers):
            layers.append(DenseLayer(in_channels, self.growth_rate))
            in_channels += self.growth_rate

        self.dense_block = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.dense_block(x)

class TransitionLayer(nn.Module):
    def __init__(self, in_channels, compression_factor=0.5):
        super(TransitionLayer, self).__init__()
        out_channels = int(in_channels * compression_factor)

        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, bias=False)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2, padding=0)
    
    def forward(self, x):
        x = self.bn(x)
        x = self.relu(x)
        x = self.conv(x)
        x = self.pool(x)
        return x
    
class DenseNet(nn.Module):
    def __init__(self, image_channels, layers, growth_rate=32, compression_factor=0.5, num_classes=100):
        super(DenseNet, self).__init__()
        self.features = []
        self.features.append(
            nn.Conv2d(image_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        )
        self.features.append(nn.BatchNorm2d(64))
        self.features.append(nn.ReLU(inplace=True))
        self.features.append(nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

        num_channels = 64
        for i, num_layers in enumerate(layers):
            dense_block = DenseBlock(num_layers, num_channels, growth_rate=growth_rate)
            self.features.append(dense_block)
            num_channels += num_layers * growth_rate

            if i != len(layers) - 1:
                transition_layer = TransitionLayer(num_channels, compression_factor=compression_factor)
                self.features.append(transition_layer)
                num_channels = int(num_channels * compression_factor)
        
        self.features.append(nn.BatchNorm2d(num_channels))
        self.features.append(nn.AdaptiveAvgPool2d((1,1)))
        self.features = nn.Sequential(*self.features)
        self.classifier = nn.Linear(num_channels, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

def DenseNet121(img_channels, num_classes):
    return DenseNet(image_channels=img_channels, layers=[6, 12, 24, 16], growth_rate=32, compression_factor=0.5, num_classes=num_classes)

def DenseNet169(img_channels, num_classes):
    return DenseNet(image_channels=img_channels, layers=[6, 12, 32, 32], growth_rate=32, compression_factor=0.5, num_classes=num_classes)

def DenseNet201(img_channels, num_classes):
    return DenseNet(image_channels=img_channels, layers=[6, 12, 48, 32], growth_rate=32, compression_factor=0.5, num_classes=num_classes)

def DenseNet264(img_channels, num_classes):
    return DenseNet(image_channels=img_channels, layers=[6, 12, 64, 48], growth_rate=32, compression_factor=0.5, num_classes=num_classes)

## ResNet

In [ ]:
class block(nn.Module):
    def __init__(self, in_channels, out_channels, identity_downsample=None, stride=1):
        super(block, self).__init__()
        self.expansion = 4
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels*self.expansion, kernel_size=1, stride=1, padding=0)
        self.bn3 = nn.BatchNorm2d(out_channels*self.expansion)
        self.relu = nn.ReLU()
        self.identity_downsample = identity_downsample

    def forward(self, x):
        identity = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv3(x)
        x = self.bn3(x)

        if self.identity_downsample is not None:
            identity = self.identity_downsample(identity)
        
        x += identity
        x = self.relu(x)
        return x
    
class ResNet(nn.Module):
    def __init__(self, block, layers, image_channels, num_classes):
        super(ResNet, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(image_channels, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet Layers
        self.layer1 = self._make_layer(block, layers[0], out_channels=64, stride=1)
        self.layer2 = self._make_layer(block, layers[1], out_channels=128, stride=2)
        self.layer3 = self._make_layer(block, layers[2], out_channels=256, stride=2)
        self.layer4 = self._make_layer(block, layers[3], out_channels=512, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512*4, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fc(x)

        return x

    def _make_layer(self, block, num_residual_blocks, out_channels, stride):
        identity_downsample = None
        layers = []

        if stride != 1 or self.in_channels != out_channels * 4:
            identity_downsample = nn.Sequential(nn.Conv2d(self.in_channels, out_channels*4, kernel_size=1,
                                                          stride=stride),
                                                nn.BatchNorm2d(out_channels*4))
        
        layers.append(block(self.in_channels, out_channels, identity_downsample, stride))
        self.in_channels = out_channels * 4

        for i in range(num_residual_blocks - 1):
            layers.append(block(self.in_channels, out_channels))
        
        return nn.Sequential(*layers)
    
def ResNet50(img_channels=3, num_classes=1000):
    return ResNet(block, [3, 4, 6, 3], img_channels, num_classes)
    
def ResNet101(img_channels=3, num_classes=1000):
    return ResNet(block, [3, 4, 23, 3], img_channels, num_classes)
    
def ResNet152(img_channels=3, num_classes=1000):
    return ResNet(block, [3, 8, 36, 3], img_channels, num_classes)

# Train

In [ ]:
# Clean CUDA
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def train(model_name):
    """
    Train a DenseNet model on the butterfly dataset.
    Args:
        model_name (str): One of "densenet121", "densenet169", "densenet201", "densenet264"
    """
    # Config
    set_seed(42)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
        
    # Datasets & loaders
    train_datasets = ButterflyDataset(
        root=DATASET_PATH, 
        split="train",
        transform=build_transforms(DATA_CFG["image_size"], train=True))
    test_datasets = ButterflyDataset(
        root=DATASET_PATH, 
        split="test",
        transform=build_transforms(DATA_CFG["image_size"], train=False))
    
    train_loader = DataLoader(train_datasets,
                              batch_size=DATA_CFG["batch_size"], 
                              shuffle=True, 
                              num_workers=DATA_CFG["num_workers"],
                              drop_last=True)
    test_loader = DataLoader(test_datasets,
                             batch_size=DATA_CFG["batch_size"], 
                             shuffle=False, 
                             num_workers=DATA_CFG["num_workers"],
                             drop_last=True)

    # Model, loss, optimizer
    if model_name == "densenet121":
        model = DenseNet121(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        MODEL_CFG = DENSENET_CFG
    elif model_name == "densenet169":
        model = DenseNet169(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        MODEL_CFG = DENSENET_CFG
    elif model_name == "densenet201":
        model = DenseNet201(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        MODEL_CFG = DENSENET_CFG
    elif model_name == "densenet264":
        model = DenseNet264(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        MODEL_CFG = DENSENET_CFG
    elif model_name == "resnet50":
        model = ResNet50(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        MODEL_CFG = RESNET_CFG
    elif model_name == "resnet101":
        model = ResNet101(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        MODEL_CFG = RESNET_CFG
    elif model_name == "resnet152":
        model = ResNet152(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
        MODEL_CFG = RESNET_CFG

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(
        model.parameters(),
        lr=float(MODEL_CFG.get("lr", 0.001)),
        momentum=float(MODEL_CFG.get("momentum", 0.9)),
        weight_decay=float(MODEL_CFG.get("weight_decay", 1e-4))
    )
    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=int(MODEL_CFG.get("lr_step_size", 7)),
        gamma=float(MODEL_CFG.get("lr_gamma", 0.1))
    )

    # Checkpoint
    num_epochs = int(MODEL_CFG.get("epochs", 50))
    output_dir = Path("/kaggle/working/outputs/checkpoints")
    output_dir.mkdir(parents=True, exist_ok=True)

    start_epoch = 1
    best_acc = 0.0

    loss_history = []
    train_acc_history = []
    test_acc_history = []
    epoch_times = []

    if MODEL_CFG.get("start_from", None) is not None and not isinstance(MODEL_CFG.get("start_from", None), str):
        ckpt_epoch = MODEL_CFG["start_from"]
        ckpt_path = MODEL_PATH / f"{model_name}_epoch_{ckpt_epoch}.pth"

        checkpoint = torch.load(ckpt_path, map_location=device)

        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])

        best_acc = checkpoint.get("best_acc", 0.0)

        loss_history = checkpoint.get("loss_history", [])
        train_acc_history = checkpoint.get("train_acc_history", [])
        test_acc_history = checkpoint.get("test_acc_history", [])
        epoch_times = checkpoint.get("epoch_times", [])

        start_epoch = checkpoint["epoch"] + 1

        print(f"Resumed from epoch {start_epoch}")

    # Training Loop
    for epoch in range(start_epoch, num_epochs+1):
        start_time = time.time()

        # Training
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        for images, labels in tqdm(train_loader, desc=f"[Train] Epoch {epoch}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)
        
        epoch_loss = running_loss / total_train
        train_acc = correct_train / total_train
        loss_history.append(epoch_loss)
        train_acc_history.append(train_acc)

        # Testing
        model.eval()
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc=f"[Test] Epoch {epoch}/{num_epochs}"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                correct_test += (preds == labels).sum().item()
                total_test += labels.size(0)
            
        test_acc = correct_test / total_test
        test_acc_history.append(test_acc)

        epoch_time = time.time() - start_time
        epoch_times.append(epoch_time)

        print(f"Epoch {epoch} | Loss: {epoch_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Test Acc: {test_acc*100:.2f}% | Time: {epoch_time:.2f}s")
        
        # Save plots
        save_training_plots(
            model_name=model_name,
            loss_history=loss_history,
            train_acc_history=train_acc_history,
            test_acc_history=test_acc_history,
            epoch_times=epoch_times,
            output_dir="outputs/plots"
        )

        if test_acc > best_acc:
            best_acc = test_acc
            # best_ckpt_path = output_dir / f"{model_name}_best.pth"
            # torch.save(ckpt, best_ckpt_path)
            # print(f"Saved best model to {best_ckpt_path}")

        # Save checkpoint
        ckpt = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_acc": best_acc,
        
            # histories
            "loss_history": loss_history,
            "train_acc_history": train_acc_history,
            "test_acc_history": test_acc_history,
            "epoch_times": epoch_times,
        }

        ckpt_path = output_dir / f"{model_name}_epoch_{epoch}.pth"
        torch.save(ckpt, ckpt_path)

        scheduler.step()

    print("\nTraining Summary")
    print(f"Best Test Accuracy: {best_acc*100:.2f}%")
    print(f"Total time: {sum(epoch_times):.2f} seconds")
    print(f"Avg time/epoch: {np.mean(epoch_times):.2f} seconds")
    print(f"Min epoch time: {np.min(epoch_times):.2f} seconds")
    print(f"Max epoch time: {np.max(epoch_times):.2f} seconds")

if __name__ == "__main__":
    model_name = "densenet121"
    train(model_name)

# Inference

In [ ]:
def inference(params_path, topk=(1,5)):
    model_name = params_path.split("_")[0]
    # Setup
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device, "\n")

    # Create output directory for plots
    plots_dir = Path(f"outputs/plots/{model_name}")
    plots_dir.mkdir(parents=True, exist_ok=True)

    # Create output directory for metrics
    metric_dir = Path("outputs/metrics")
    metric_dir.mkdir(parents=True, exist_ok=True)

    # Data
    test_dataset = ButterflyDataset(
        root=DATASET_PATH, 
        split="valid",  
        transform=build_transforms(DATA_CFG["image_size"], train=False)
    )
    test_loader = DataLoader(
        test_dataset, batch_size=64, shuffle=False, num_workers=1
    )

    idx_to_class = test_dataset.idx_to_class

    # Model, loss, optimizer
    if model_name == "densenet121":
        model = DenseNet121(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
    elif model_name == "densenet169":
        model = DenseNet169(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
    elif model_name == "densenet201":
        model = DenseNet201(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
    elif model_name == "densenet264":
        model = DenseNet264(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
    elif model_name == "resnet50":
        model = ResNet50(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
    elif model_name == "resnet101":
        model = ResNet101(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)
    elif model_name == "resnet152":
        model = ResNet152(img_channels=3, num_classes=DATA_CFG.get("num_classes", 100)).to(device)

    
    ckpt_path = MODEL_PATH / params_path
    checkpoint = torch.load(ckpt_path, map_location=device)
    state_dict = checkpoint["model_state"]
    
    new_state_dict = {}
    for k, v in state_dict.items():
        new_key = k.replace("module.", "")
        new_state_dict[new_key] = v
    
    model.load_state_dict(new_state_dict)
    model.eval()
    
    print_model_size(model, device)

    # Metrics Tracking
    total = 0
    topk_correct = [0] * len(topk)
    confusion_counter = Counter()      # (true, pred)
    per_class_total = Counter()        # true
    per_class_correct = Counter()      # true & correct

    # Inference Loop
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"[Inference]"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            probs = torch.softmax(logits, dim=1)

            # Top-k accuracy
            for i, k in enumerate(topk):
                topk_preds = torch.topk(probs, k, dim=1).indices
                topk_correct[i] += (
                    topk_preds == labels.unsqueeze(1)
                ).any(dim=1).sum().item()

            # Top-1 predictions
            preds = torch.argmax(probs, dim=1)

            for t, p in zip(labels.cpu().numpy(), preds.cpu().numpy()):
                per_class_total[t] += 1
                if t == p:
                    per_class_correct[t] += 1
                else:
                    confusion_counter[(t, p)] += 1

            total += labels.size(0)

    # Print accuracy
    print("\nAccuracy:")
    for i, k in enumerate(topk):
        acc = topk_correct[i] / total
        print(f"Top-{k}: {acc:.4f}")

    print(f"Highest test accuracy: {max(checkpoint.get('test_acc_history', [0])):.4f}")
    print(f"at epoch: {checkpoint.get('test_acc_history', []).index(max(checkpoint.get('test_acc_history', [0])))}")

    # Confusion analysis
    most_confused = confusion_counter.most_common(10)

    print("\nTop 10 most confused class pairs (true -> predicted):")
    for (t, p), count in most_confused:
        print(f"{idx_to_class[t]} -> {idx_to_class[p]} : {count}")

    if most_confused:
        # Bar plot for top 10 most confused
        labels_plot = [
            f"{idx_to_class[t]}->{idx_to_class[p]}"
            for (t, p), _ in most_confused
        ]
        counts = [c for _, c in most_confused]

        plt.figure(figsize=(10, 5))
        plt.bar(range(len(counts)), counts)
        plt.xticks(range(len(counts)), labels_plot, rotation=45)
        plt.ylabel("Count")
        plt.title("Top 10 Most Confused Class Pairs")
        plt.tight_layout()

        plot_path = plots_dir / f"{model_name}_most_confused_pairs.png"
        plt.savefig(plot_path)
        plt.close()
        print(f"\nConfusion plot saved to: {plot_path}")

        # Automatic Top-10 Confused Image Grid

        fig, axes = plt.subplots(5, 4, figsize=(18, 20))
        axes = axes.reshape(5, 4)

        for idx, ((t, p), _) in enumerate(most_confused):
            row = idx // 2
            col = (idx % 2) * 2

            true_name = idx_to_class[t]
            pred_name = idx_to_class[p]

            # Get filepaths for true and predicted classes
            t_imgs = [img_path for img_path, label in test_dataset.samples if label == t]
            p_imgs = [img_path for img_path, label in test_dataset.samples if label == p]

            # Sample up to 2 images per class safely
            t_sample = random.sample(t_imgs, min(2, len(t_imgs)))
            p_sample = random.sample(p_imgs, min(2, len(p_imgs)))

            # Fill 2 columns (true vs predicted)
            for i in range(2):
                if i < len(t_sample):
                    img_path = DATASET_PATH / t_sample[i]
                    axes[row, col].imshow(Image.open(img_path).convert("RGB"))
                    axes[row, col].set_title(f"True: {true_name}", fontsize=9)
                    axes[row, col].axis("off")

                if i < len(p_sample):
                    img_path = DATASET_PATH / p_sample[i]
                    axes[row, col + 1].imshow(Image.open(img_path).convert("RGB"))
                    axes[row, col + 1].set_title(f"Pred: {pred_name}", fontsize=9)
                    axes[row, col + 1].axis("off")

        plt.tight_layout()
        sample_img_path = plots_dir / f"{model_name}_most_confused_pairs_samples.png"
        plt.savefig(sample_img_path)
        plt.close()
        print(f"\nSample images of confused pairs saved to: {sample_img_path}")

    # Per-class accuracy CSV
    class_accuracy = []
    for cls in per_class_total:
        acc = per_class_correct[cls] / per_class_total[cls]
        class_accuracy.append(
            (cls, idx_to_class[cls], acc, per_class_correct[cls], per_class_total[cls])
        )

    # Sort high -> low accuracy
    class_accuracy.sort(key=lambda x: x[2], reverse=True)

    csv_path = metric_dir / f"{model_name}_per_class_accuracy.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["class_id", "class_name", "accuracy", "correct", "total"])
        for cls, name, acc, correct, total_cls in class_accuracy:
            writer.writerow([cls, name, f"{acc:.4f}", correct, total_cls])

    print(f"\nPer-class accuracy CSV saved to: {csv_path}")

    # Save plots
    save_training_plots(
        model_name=model_name,
        loss_history=checkpoint.get("loss_history", []),
        train_acc_history=checkpoint.get("train_acc_history", []),
        test_acc_history=checkpoint.get("test_acc_history", []),
        epoch_times=checkpoint.get("epoch_times", []),
        output_dir=plots_dir
    )

    print("\nPlots saved")

if __name__ == "__main__":
    param_path = "densenet121_epoch_50.pth"
    inference(param_path)
    summarize_checkpoint_times(param_path)